# TFM — Parque de Vehículos (DGT) vs. Renta de los Hogares (INE)
## Fase 0 — Adquisición e Integración de Fuentes

Este notebook documenta la **primera fase** del pipeline: la obtención de las fuentes
de datos en bruto y su integración en un único conjunto de trabajo. **No** incluye la
limpieza profunda, el feature engineering ni el análisis exploratorio (EDA), que se
abordan en notebooks posteriores.

**Fuentes utilizadas**

1. **DGT — Microdatos del parque de vehículos** (marzo 2026). Censo de vehículos
   matriculados, un registro por vehículo, con ficha técnica completa.
   Aporta la dimensión *transaccional* del proyecto.
2. **INE — Atlas de Distribución de Renta de los Hogares** (datos de 2023, publicados
   en octubre de 2025). Indicadores de renta por municipio.
   Aporta el *contexto socioeconómico*.

Ambas son fuentes de **datos abiertos** procedentes de **canales distintos**, y se
integran mediante el **código INE de municipio (5 dígitos)** como clave común.

**Tabla maestra de referencia**

3. **INE — Relación de provincias y sus códigos.** Diccionario oficial código↔nombre
   de provincia (denominaciones oficiales, p. ej. *Alicante/Alacant*). Se emplea para
   etiquetar de forma legible los códigos de provincia; no aporta datos analíticos.




---
## 1. Configuración e importaciones

Se centralizan rutas y parámetros al inicio para facilitar la reproducibilidad.
Ajusta las rutas de los ficheros a tu estructura local si es necesario.


In [1]:
# ==============================================================================
# 1. LIBRERÍAS DE LA BIBLIOTECA ESTÁNDAR
# ==============================================================================
import sys
from pathlib import Path

# ==============================================================================
# 2. LIBRERÍAS DE TERCEROS (Data Science & Analytics)
# ==============================================================================
import pandas as pd

# Añadir el directorio de código fuente ('src') al PATH para poder importar módulos locales
sys.path.append("../src")


# ==============================================================================
# 3. MÓDULOS PROPIOS (Trabajo de Fin de Máster - TFM)
# ==============================================================================
import tfm_io        as ti      # Funciones de lectura, escritura y gestión de archivos (I/O)

# ==============================================================================
# 4. RUTAS DE FICHEROS (Ajustar a la ubicación local) 
# ==============================================================================
DGT_NACIONAL = Path("../data/raw/parque_vehiculos_202603.txt")   # Microdatos DGT (nacional)
INE_RENTA    = Path("../data/raw/30824.csv")                     # Atlas de Renta INE (municipal)
INE_PROV     = Path("../data/raw/codprov.xls")                   # Códigos de provincia INE

DIR_RAW       = Path("../data/raw")
DIR_PROCESSED = Path("../data/processed")
DIR_PROCESSED.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# 5. PROVINCIAS DEL ESTUDIO (Códigos INE de 2 dígitos) 
# 28 Madrid | 08 Barcelona | 46 Valencia | 03 Alicante | 41 Sevilla
# ==============================================================================
TOP5_PREFIJOS = ["28", "08", "46", "03", "41"]

# ==============================================================================
# 6. CONTROL DE VALORES ANÓNIMOS (DGT)
# La DGT enmascara con el carácter '¡' (\xa1 en latin-1) los valores anonimizados
# (marcas/modelos con <= 5 ocurrencias). Se tratan como NA en la lectura.
# ==============================================================================
NA_VALUES = ["\xa1", ""]

**INE — Relación de provincias y sus códigos.** Diccionario oficial código↔nombre

Cargamos el listado oficial de provincias del INE (https://www.ine.es/daco/daco42/codmun/cod_provincia.htm) y construimos diccionario

In [2]:
# Carga CSV del listado oficial de provincias del INE (fuente reproducible)
df_prov = ti.cargar_excel(
    INE_PROV, descripcion="provincias INE",
    engine="xlrd", sheet_name=0, skiprows=2,
    header=None, names=["codigo", "nombre"], dtype=str,
)

Cargado (provincias INE): 52 filas x 2 columnas


In [3]:
# Estandarización de códigos provinciales. 
# Limpiar espacios invisibles y forzar el formato de 2 dígitos
# Limpiar espacios accidentales al principio o final de los nombres
# Eliminar filas con códigos o nombres vacíos (después de la limpieza)

df_prov["codigo"] = df_prov["codigo"].str.strip().str.zfill(2)
df_prov["nombre"] = df_prov["nombre"].str.strip()
df_prov = df_prov.dropna(subset=["codigo", "nombre"])

CODIGOS_PROVINCIA = dict(zip(df_prov["codigo"], df_prov["nombre"]))

print(df_prov)
print(f"Provincias cargadas: {len(CODIGOS_PROVINCIA)}")

   codigo                  nombre
0      02                Albacete
1      03        Alicante/Alacant
2      04                 Almería
3      01             Araba/Álava
4      33                Asturias
5      05                   Ávila
6      06                 Badajoz
7      07          Balears, Illes
8      08               Barcelona
9      48                 Bizkaia
10     09                  Burgos
11     10                 Cáceres
12     11                   Cádiz
13     39               Cantabria
14     12      Castellón/Castelló
15     13             Ciudad Real
16     14                 Córdoba
17     15               Coruña, A
18     16                  Cuenca
19     20                Gipuzkoa
20     17                  Girona
21     18                 Granada
22     19             Guadalajara
23     21                  Huelva
24     22                  Huesca
25     23                    Jaén
26     24                    León
27     25                  Lleida
28     27     

---
## 2. Fuente DGT — Análisis de representatividad y selección de provincias

Antes de filtrar, se justifica **con datos** el alcance geográfico. Se carga el fichero
nacional leyendo **solo las columnas necesarias** (`usecols`) para minimizar el uso de
memoria, ya que el fichero completo contiene ~39 millones de registros.

El objetivo es comprobar qué porcentaje del parque nacional de turismos concentran las
provincias seleccionadas.


In [4]:
# Lectura ligera: solo las columnas necesarias para el analisis de representatividad
df_nacional = ti.cargar_csv(
    DGT_NACIONAL, descripcion="DGT",
    sep="|", encoding="latin-1", low_memory=False,
    dtype=str, na_values=NA_VALUES,
    usecols=["PROVINCIA", "TIPO_DGT", "FEC_PRIM_MATR"],
)

print("\nDistribucion por tipo de vehiculo:")
print(df_nacional["TIPO_DGT"].value_counts().head(10).to_string())

Cargado (DGT): 38,945,977 filas x 3 columnas

Distribucion por tipo de vehiculo:
TIPO_DGT
TURISMOS                  25994390
MOTOCICLETAS               4560366
FURGONETAS                 2812245
CAMIONES                   2462611
CICLOMOTORES               1592778
OTROS vehiculos             609678
R Y S                       574897
TRACTORES INDUSTRIALES      269970
AUTOBUSES                    69042


In [5]:
# Filtrar solo turismos y calcular distribución provincial
df_turismos_nac = df_nacional[df_nacional["TIPO_DGT"] == "TURISMOS"].copy()
total_turismos_esp = len(df_turismos_nac)
print(f"Total turismos España: {total_turismos_esp:,}")

parque_prov = (
    df_turismos_nac
    .groupby("PROVINCIA", observed=True).size()
    .reset_index(name="num_turismos")
    .sort_values("num_turismos", ascending=False)
    .reset_index(drop=True)
)

# Porcentajes (el reset_index previo garantiza un cumsum correcto)
parque_prov["pct_nacional"]  = (parque_prov["num_turismos"] / total_turismos_esp * 100).round(2)
parque_prov["pct_acumulado"] = parque_prov["pct_nacional"].cumsum().round(2)
parque_prov["nombre_provincia"] = parque_prov["PROVINCIA"].map(CODIGOS_PROVINCIA)

print("\nTop 10 provincias por parque de turismos:")
print(parque_prov[["nombre_provincia", "num_turismos", "pct_nacional", "pct_acumulado"]]
      .head(10).to_string(index=False))

pct_top5 = parque_prov.head(5)["pct_nacional"].sum()
print(f"\nLas 5 provincias con mayor parque concentran el {pct_top5:.2f}% del parque nacional de turismos.")

Total turismos España: 25,994,390

Top 10 provincias por parque de turismos:
 nombre_provincia  num_turismos  pct_nacional  pct_acumulado
           Madrid       4209735         16.19          16.19
        Barcelona       2407927          9.26          25.45
Valencia/València       1337767          5.15          30.60
 Alicante/Alacant       1125379          4.33          34.93
          Sevilla       1007950          3.88          38.81
           Málaga        946045          3.64          42.45
           Murcia        856941          3.30          45.75
   Balears, Illes        758920          2.92          48.67
      Palmas, Las        669809          2.58          51.25
        Coruña, A        654769          2.52          53.77

Las 5 provincias con mayor parque concentran el 38.81% del parque nacional de turismos.


> 🟢 **Resultado.** Las cinco provincias seleccionadas — **Madrid, Barcelona, Valencia,
> Alicante y Sevilla** — concentran el **38,81 %** del parque nacional de turismos
> (10.088.758 sobre 25.994.390). La selección se justifica por tres criterios:
>
> - **Peso en el parque nacional**: casi 4 de cada 10 turismos del país, lo que aporta
>   volumen y solidez estadística al análisis.
> - **Diversidad de perfiles**: incluye la capital económica (Madrid), una gran
>   metrópolis industrial (Barcelona), núcleos del arco mediterráneo (Valencia, Alicante)
>   y la mayor ciudad del sur (Sevilla), criterio cualitativo de partida.
> - **Viabilidad computacional**: trabajar con las 5 mayores mantiene el volumen
>   manejable; ampliar al conjunto nacional multiplicaría el tamaño de forma
>   desproporcionada para el alcance de este TFM.

---
## 3. Fuente DGT — Carga completa, filtrado y guardado

Se recarga el fichero **con todas las columnas**, se convierten las fechas al tipo
`datetime` y se aplica el filtro definitivo:

- Solo **turismos** (`TIPO_DGT == "TURISMOS"`).
- Solo las **5 provincias** del estudio.
- Matriculados **desde 2010** (coherente con el mercado de vehículos en circulación).


In [6]:
# Carga completa con todas las columnas
df_full = ti.cargar_csv(
    DGT_NACIONAL, descripcion="DGT",
    sep="|", encoding="latin-1", low_memory=False,
    dtype=str, na_values=NA_VALUES,
)

Cargado (DGT): 38,945,977 filas x 45 columnas


In [7]:
# Conversion de fechas (formato DGT: DD/MM/AAAA)
for col in ["FECHA_MATR", "FEC_PRIM_MATR"]:
    df_full[col] = pd.to_datetime(df_full[col], format="%d/%m/%Y", errors="coerce")

# Nombre de provincia legible
df_full["nombre_provincia"] = df_full["PROVINCIA"].map(CODIGOS_PROVINCIA)

# Filtro definitivo: turismos + 5 provincias + matriculados desde 2010
df_trabajo = df_full[
    (df_full["TIPO_DGT"] == "TURISMOS") &
    (df_full["PROVINCIA"].isin(TOP5_PREFIJOS)) &
    (df_full["FEC_PRIM_MATR"].dt.year >= 2010)
].copy()

print(f"\nDataset DGT de trabajo: {len(df_trabajo):,} filas x {len(df_trabajo.columns)} columnas")
print("\nReparto por provincia:")
print(df_trabajo.groupby("nombre_provincia", observed=True).size()
      .sort_values(ascending=False).to_string())


Dataset DGT de trabajo: 6,613,034 filas x 46 columnas

Reparto por provincia:
nombre_provincia
Madrid               3066219
Barcelona            1523487
Valencia/València     808152
Alicante/Alacant      647414
Sevilla               567762


> 🟢 Se guarda el subconjunto DGT en **formato Parquet**. Frente a CSV, Parquet ocupa
> mucho menos espacio (~5-10×), conserva los tipos de dato y carga más rápido —
> una decisión técnica justificada dado el volumen (varios millones de filas).

In [8]:
# guardado del dataset para análisis posteriores (formato parquet optimizado)
ruta_dgt = DIR_PROCESSED / "dgt_top5_turismos_2010_202603.parquet"
ti.guardar_parquet(df_trabajo, ruta_dgt)


Guardado: ..\data\processed\dgt_top5_turismos_2010_202603.parquet  (6,613,034 filas, 188.3 MB)


> 🟢 *(Opcional)* Muestra aleatoria **estratificada por provincia** en CSV, solo para
> inspección visual rápida en Excel. No es el fichero de trabajo. `random_state=42`
> garantiza que la muestra sea reproducible.

In [9]:
# generación de una muestra representativa de 10k filas (2k por provincia) 
muestra = (
    df_trabajo
    .groupby("nombre_provincia", observed=True)
    .sample(n=2000, random_state=42)
    .reset_index(drop=True)
)

#muestra.to_csv(DIR_RAW / "dgt_top5_muestra_10k.csv",
#               index=False, encoding="utf-8-sig", sep=";")
#print(f"Muestra guardada: {len(muestra):,} filas")

# guardado de la muestra en formato CSV
ruta=DIR_PROCESSED / "dgt_top5_muestra_10k.csv"
ti.guardar_csv_es(muestra, ruta)

# Verificación rápida de la distribución provincial en la muestra
print(muestra["nombre_provincia"].value_counts().to_string())



Guardado: ..\data\processed\dgt_top5_muestra_10k.csv  (10,000 filas, 2.3 MB)
nombre_provincia
Alicante/Alacant     2000
Barcelona            2000
Madrid               2000
Sevilla              2000
Valencia/València    2000


---
## 4. Fuente INE — Atlas de Renta a nivel municipal

El fichero del INE viene en formato **largo**: cada fila es un municipio × indicador ×
año, y el nivel geográfico está separado en tres columnas (`Municipios`, `Distritos`,
`Secciones`). La columna `Municipios` trae el **código INE de 5 dígitos** pegado al
nombre (p. ej. `28115 Pozuelo de Alarcón`).

Pasos de procesado:
1. Detectar automáticamente el nombre de la columna de indicadores (varía entre tablas).
2. Quedarse **solo con el nivel municipio** (`Distritos` y `Secciones` vacíos).
3. Filtrar al **último año disponible (2023)**.
4. Separar **código** y **nombre** del municipio.
5. Convertir el importe a numérico (el INE usa `.` como separador de miles, y un `.`
   aislado significa *dato no disponible* → `NaN`).
6. **Pivotar** los indicadores a columnas (una fila por municipio).


In [10]:
# Lectura del Atlas de Renta (CSV separado por ';')
df_ine = ti.cargar_csv(
    INE_RENTA, descripcion="Atlas de Renta",
    sep=";", encoding="utf-8", dtype=str,
)

print("Columnas:", df_ine.columns.tolist())

# Deteccion automatica de la columna de indicadores (las fijas son siempre las mismas)
FIJAS = {"Municipios", "Distritos", "Secciones", "Periodo", "Total"}
COL_IND = [c for c in df_ine.columns if c not in FIJAS][0]
print(f"Columna de indicadores detectada: '{COL_IND}'")

Cargado (Atlas de Renta): 3,009,312 filas x 6 columnas
Columnas: ['Municipios', 'Distritos', 'Secciones', 'Indicadores de renta media', 'Periodo', 'Total']
Columna de indicadores detectada: 'Indicadores de renta media'


In [11]:
# 1-2. Solo nivel municipio (sin distrito/sección), último año
df_muni = df_ine[df_ine["Distritos"].isna() & df_ine["Secciones"].isna()].copy()
df_muni = df_muni[df_muni["Periodo"] == "2023"].copy()

# 3. Código (5 dígitos) y nombre limpio del municipio
df_muni["cod_municipio"]    = df_muni["Municipios"].str.extract(r"^(\d{5})")
df_muni["nombre_municipio"] = df_muni["Municipios"].str.replace(r"^\d{5}\s+", "", regex=True)

# 4. Importe a numérico: quitar separador de miles; '.' aislado -> NaN
df_muni["valor"] = (
    df_muni["Total"].str.replace(".", "", regex=False).replace("", pd.NA)
)
df_muni["valor"] = pd.to_numeric(df_muni["valor"], errors="coerce")

# 5. Pivotar indicadores a columnas -> una fila por municipio
df_renta = (
    df_muni
    .pivot_table(index=["cod_municipio", "nombre_municipio"],
                 columns=COL_IND, values="valor", aggfunc="first")
    .reset_index()
)
df_renta.columns.name = None

# 6. Filtrar a las 5 provincias del estudio
df_renta_top5 = df_renta[df_renta["cod_municipio"].str[:2].isin(TOP5_PREFIJOS)].copy()

print(f"Municipios nacionales : {len(df_renta):,}")
print(f"Municipios en las 5 provincias: {len(df_renta_top5)}")
print("\nReparto por provincia:")
print(df_renta_top5["cod_municipio"].str[:2].value_counts().to_string())

Municipios nacionales : 8,059
Municipios en las 5 provincias: 993

Reparto por provincia:
cod_municipio
08    307
46    263
28    179
03    138
41    106


> 🟢 **Validación de la lectura.** Se contrasta contra una cifra oficial conocida:
> Pozuelo de Alarcón fue en 2023 el municipio de más de 2.000 habitantes con mayor renta
> neta media por persona de España, **30.524 €** (nota de prensa del INE). Si la cifra
> coincide, confirma que la conversión numérica y el tratamiento del separador de miles
> son correctos.

In [12]:
pozuelo = df_renta_top5[df_renta_top5["nombre_municipio"].str.contains("Pozuelo de Alarcón", na=False)]
print(pozuelo.to_string(index=False))

cod_municipio   nombre_municipio  Media de la renta por unidad de consumo  Mediana de la renta por unidad de consumo  Renta bruta media por hogar  Renta bruta media por persona  Renta neta media por hogar  Renta neta media por persona
        28115 Pozuelo de Alarcón                                  49811.0                                    37450.0                     138851.0                        44016.0                     96290.0                       30524.0


---
## 5. Integración de fuentes (merge)

Ambas fuentes comparten el **código INE de municipio de 5 dígitos**:
`MUNICIPIO` (DGT) ↔ `cod_municipio` (INE). La relación es **muchos-a-uno**
(muchos vehículos por municipio), por lo que se usa un **left join** que conserva
todos los vehículos.

> 🟢 **Desfase temporal entre fuentes.** El parque DGT es de marzo 2026 y la renta INE de
> 2023 (último año publicado; el INE va con ~2 años de retardo por usar datos fiscales
> consolidados). El cruce es válido porque la renta municipal es estructural y de baja
> volatilidad. Justificación ampliada en el README.

Primero se verifica la compatibilidad de las llaves (mismo formato, mismo tipo).


In [13]:
# Verificación de compatibilidad de llaves
print("DGT  MUNICIPIO     :", df_trabajo["MUNICIPIO"].dropna().head(3).tolist(),
      "| tipo:", df_trabajo["MUNICIPIO"].dtype)
print("INE  cod_municipio :", df_renta_top5["cod_municipio"].head(3).tolist(),
      "| tipo:", df_renta_top5["cod_municipio"].dtype)

cods_ine = set(df_renta_top5["cod_municipio"])
cods_dgt = set(df_trabajo["MUNICIPIO"].dropna().unique())
print(f"\nMunicipios DGT distintos      : {len(cods_dgt)}")
print(f"De ellos, presentes en INE    : {len(cods_dgt & cods_ine)}")
print(f"Ausentes en INE (serían NaN)  : {len(cods_dgt - cods_ine)}")

DGT  MUNICIPIO     : ['03056', '03018', '03018'] | tipo: object
INE  cod_municipio : ['03001', '03002', '03003'] | tipo: object

Municipios DGT distintos      : 273
De ellos, presentes en INE    : 273
Ausentes en INE (serían NaN)  : 0


In [14]:
# Left join: conservar todos los vehículos
df_final = df_trabajo.merge(
    df_renta_top5,
    how="left",
    left_on="MUNICIPIO",
    right_on="cod_municipio",
)

print(f"Filas tras merge : {len(df_final):,}")
print(f"Columnas         : {len(df_final.columns)}")

pct_con_renta = df_final["Renta neta media por persona"].notna().mean() * 100
print(f"Vehículos con renta asignada: {pct_con_renta:.2f}%")

Filas tras merge : 6,613,034
Columnas         : 54
Vehículos con renta asignada: 83.21%


> 🟢 **Diagnóstico de la cobertura del cruce.** Conviene entender por qué no todos los
> vehículos reciben renta. El **documento oficial de interfaz** de la DGT lo aclara: la
> variable `MUNICIPIO` se **suprime** (queda vacía) para los vehículos domiciliados en
> **municipios de menos de 10.000 habitantes**, por anonimización. Por tanto:
>
> - **Causa 1** — el vehículo está en un municipio de <10.000 hab. → sin `MUNICIPIO`.
> - **Causa 2** — el municipio existe pero su renta está **suprimida** en el INE (`NaN`).

In [15]:
sin_muni = df_final["MUNICIPIO"].isna().sum()
print(f"Sin MUNICIPIO en DGT (causa 1): {sin_muni:,} ({sin_muni/len(df_final)*100:.2f}%)")

con_muni_sin_renta = df_final[
    df_final["MUNICIPIO"].notna() &
    df_final["Renta neta media por persona"].isna()
]
print(f"Con municipio pero renta NaN (causa 2): {len(con_muni_sin_renta):,} "
      f"({len(con_muni_sin_renta)/len(df_final)*100:.2f}%)")

Sin MUNICIPIO en DGT (causa 1): 1,110,202 (16.79%)
Con municipio pero renta NaN (causa 2): 0 (0.00%)


> 🟢 **Interpretación.** El faltante (~16,79 %, ~1,1 M de vehículos) procede
> **íntegramente de la causa 1**: vehículos domiciliados en municipios de menos de 10.000
> habitantes, cuyo municipio la DGT suprime por anonimización (documentado en la interfaz
> oficial del fichero de parque). El INE **no introduce ningún hueco**: siempre que hay
> municipio, existe su renta.
>
> **Implicación analítica.** El análisis renta↔parque queda acotado a los municipios de
> **≥10.000 habitantes** (los 273 presentes en el cruce). No es un error de datos, sino una
> **limitación de cobertura** de la fuente, que se documenta como tal y se gestiona con la
> bandera `tiene_municipio`.
>
> **Decisión de diseño.** Se conservan todos los vehículos (el faltante se gestiona en la
> fase de limpieza) y se añade la bandera explícita `tiene_municipio`, que permite filtrar
> de forma transparente en los análisis que dependan de la renta municipal.

In [16]:
df_final["tiene_municipio"] = df_final["MUNICIPIO"].notna()

print("Cobertura municipal del parque:")
print(df_final["tiene_municipio"].value_counts().to_string())
print(f"\n{df_final['tiene_municipio'].mean()*100:.2f}% de vehículos con municipio asignado")

print("\nVehículos SIN municipio, por provincia:")
sin = df_final[df_final["tiene_municipio"] == False]
print(sin["nombre_provincia"].value_counts().to_string())

Cobertura municipal del parque:
tiene_municipio
True     5502832
False    1110202

83.21% de vehículos con municipio asignado

Vehículos SIN municipio, por provincia:
nombre_provincia
Madrid               504262
Barcelona            229268
Valencia/València    180531
Alicante/Alacant     112810
Sevilla               83331


---
## 6. Guardado del dataset fusionado

Se almacena el resultado de la integración como **punto de partida congelado** de las
fases siguientes. A partir de aquí, todo el trabajo de limpieza, transformación y EDA
parte de este fichero, **sin volver a tocar las fuentes originales**.


In [17]:
# Guardado del dataset final (formato parquet optimizado)
ruta_final = DIR_PROCESSED / "dataset_fusionado.parquet"
ti.guardar_parquet(df_final, ruta_final)

Guardado: ..\data\processed\dataset_fusionado.parquet  (6,613,034 filas, 212.3 MB)


---
## Resumen de la Fase 0

| Requisito mínimo del TFM | Exigido | Conseguido |
|---|---|---|
| Filas | 50.000 | **6.613.034** |
| Columnas | 20 | **54** |
| Nº de fuentes | 2 | DGT + INE |
| Canales distintos | 2 | Open data tráfico + estadística |
| Fusión con llave común | Sí | `MUNICIPIO` ↔ `cod_municipio` |
| Tipos de dato mixtos | num / cat / fecha | Presentes |

**Próxima fase — ETL / Limpieza profunda.** Sobre `dataset_fusionado.parquet`:
gestión de nulos (incl. los ~16,79 % sin municipio), *type casting* definitivo,
estandarización de texto, y **feature engineering** (descomposición de fechas,
antigüedad del vehículo, ratios potencia/cilindrada, segmentos de renta, flags de
electrificación, etc.).
